In [ ]:
# -*- coding: utf-8 -*-


analisis_uyu_brl_colab.ipynb

Automatically generated by Colab.

# Analisis reproducible del tipo de cambio UYU/BRL

**Proyecto:** Probabilidad y Estadistica
**Tema:** Tipo de cambio entre el peso uruguayo y el real brasileno
**Variable principal:** `uyu_por_brl`

Este notebook documenta el codigo usado para cargar el dataset, validar los
datos, calcular tablas, generar graficos y aplicar los procedimientos
estadisticos incluidos en el informe.

Python se usa como herramienta computacional. Los metodos estadisticos aplicados
son descriptiva, graficos exploratorios, autocorrelacion descriptiva, correlacion
de Pearson descriptiva, prueba chi-cuadrado de bondad de ajuste e intervalos de
confianza para media y varianza.

No se genera matriz de correlacion.

## 1. Configuracion del entorno

La siguiente celda instala dependencias si se ejecuta en Google Colab. En un
entorno local ya preparado, se puede dejar comentada.


In [ ]:

# Celda opcional para Google Colab si faltan dependencias:
# !pip install pandas numpy matplotlib scipy yfinance openpyxl

from pathlib import Path
import os

import matplotlib
import numpy as np
import pandas as pd
from scipy import stats

if not os.environ.get("COLAB_RELEASE_TAG"):
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 100)
pd.set_option("display.precision", 4)

PRIMARY_COLOR = "#1f77b4"
ACCENT_COLOR = "#9467bd"
BAR_COLOR = "#4c78a8"
GRID_COLOR = "#d9e2ec"


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() or (candidate / "src").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
DATA_URL = "https://raw.githubusercontent.com/valentinzamit26/Proyecto-PYE/report-redaccion-v0.20/data/raw/dataset_uyu_brl_2005_2026.csv"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dataset_uyu_brl_2005_2026.csv"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Raiz del proyecto: {PROJECT_ROOT}")
print(f"Dataset local esperado: {DATA_PATH}")
print(f"Tablas: {TABLES_DIR}")
print(f"Figuras: {FIGURES_DIR}")



## 2. Funciones auxiliares

Estas funciones se usan durante todo el analisis para cargar datos, validar
columnas, guardar tablas y guardar figuras.


In [ ]:

REQUIRED_COLUMNS = [
    "fecha",
    "brl_por_usd",
    "uyu_por_usd",
    "brl_por_uyu",
    "uyu_por_brl",
]


def validate_columns(dataframe: pd.DataFrame) -> None:
    missing = [column for column in REQUIRED_COLUMNS if column not in dataframe.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")


def load_dataset(path: Path = DATA_PATH) -> pd.DataFrame:
    if path.exists():
        dataframe = pd.read_csv(path)
    else:
        try:
            dataframe = pd.read_csv(DATA_URL)
            path.parent.mkdir(parents=True, exist_ok=True)
            dataframe.to_csv(path, index=False)
            print(f"Dataset descargado desde GitHub y guardado en: {path}")
        except Exception:
            try:
                from google.colab import files  # type: ignore
            except ModuleNotFoundError as exc:
                raise FileNotFoundError(
                    "No se encontro el CSV local y no se pudo descargar desde GitHub."
                ) from exc

            print("Subi manualmente dataset_uyu_brl_2005_2026.csv")
            uploaded = files.upload()
            if not uploaded:
                raise FileNotFoundError("No se subio ningun archivo.")
            filename, content = next(iter(uploaded.items()))
            if not filename.lower().endswith(".csv"):
                raise ValueError(f"Se esperaba un CSV y se recibio: {filename}")
            path.write_bytes(content)
            dataframe = pd.read_csv(path)

    validate_columns(dataframe)
    dataframe["fecha"] = pd.to_datetime(dataframe["fecha"], errors="coerce")
    dataframe = dataframe.sort_values("fecha").reset_index(drop=True)
    if "retorno_diario_uyu_por_brl" not in dataframe.columns:
        dataframe["retorno_diario_uyu_por_brl"] = dataframe["uyu_por_brl"].pct_change()
    if "log_retorno_uyu_por_brl" not in dataframe.columns:
        dataframe["log_retorno_uyu_por_brl"] = np.log(dataframe["uyu_por_brl"]).diff()
    return dataframe


def clean_numeric(values: pd.Series) -> pd.Series:
    return pd.to_numeric(values, errors="coerce").dropna()


def save_table(dataframe: pd.DataFrame, filename: str) -> Path:
    path = TABLES_DIR / filename
    dataframe.to_csv(path, index=False)
    return path


def save_figure(fig: plt.Figure, filename: str) -> Path:
    path = FIGURES_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    if os.environ.get("COLAB_RELEASE_TAG"):
        plt.show()
    plt.close(fig)
    return path


def apply_common_style(ax: plt.Axes) -> None:
    ax.grid(True, color=GRID_COLOR, linewidth=0.8, alpha=0.8)
    ax.set_axisbelow(True)



## 3. Codigo opcional para reconstruir el dataset desde Yahoo Finance

El informe usa el CSV ya generado. Esta seccion deja documentado el codigo de
origen para reconstruirlo con `yfinance` si fuese necesario.


In [ ]:

START_DATE = "2005-01-01"
END_DATE = "2026-06-01"
TICKERS = {
    "uyu_por_usd": "USDUYU=X",
    "brl_por_usd": "USDBRL=X",
}


def descargar_datos_yfinance(start: str = START_DATE, end: str = END_DATE) -> pd.DataFrame:
    import yfinance as yf

    data = yf.download(
        list(TICKERS.values()),
        start=start,
        end=end,
        interval="1d",
        auto_adjust=False,
        progress=False,
    )
    if data.empty:
        raise ValueError("No se descargaron datos. Revisar conexion o tickers.")

    close = data["Close"].rename(
        columns={
            "USDUYU=X": "uyu_por_usd",
            "USDBRL=X": "brl_por_usd",
        }
    )
    return close.dropna().copy()


def crear_dataset_uyu_brl(close: pd.DataFrame) -> pd.DataFrame:
    dataset = close.copy()
    dataset["brl_por_uyu"] = dataset["brl_por_usd"] / dataset["uyu_por_usd"]
    dataset["uyu_por_brl"] = dataset["uyu_por_usd"] / dataset["brl_por_usd"]
    dataset["retorno_diario_brl_por_uyu"] = dataset["brl_por_uyu"].pct_change()
    dataset["log_retorno_brl_por_uyu"] = np.log(dataset["brl_por_uyu"]).diff()
    dataset["retorno_diario_uyu_por_brl"] = dataset["uyu_por_brl"].pct_change()
    dataset["log_retorno_uyu_por_brl"] = np.log(dataset["uyu_por_brl"]).diff()
    dataset = dataset.reset_index().rename(columns={"Date": "fecha"})
    dataset["fecha"] = pd.to_datetime(dataset["fecha"])
    return dataset


# Para reconstruir el CSV desde cero, descomentar:
# close = descargar_datos_yfinance()
# df_reconstruido = crear_dataset_uyu_brl(close)
# df_reconstruido.to_csv(DATA_PATH, index=False, encoding="utf-8-sig")



## 4. Carga y validacion inicial

Se carga el dataset principal, se valida la estructura minima y se revisan nulos,
duplicados y reciprocidad entre `uyu_por_brl` y `brl_por_uyu`.


In [ ]:

df = load_dataset(DATA_PATH)
x = clean_numeric(df["uyu_por_brl"])

validation_summary = pd.DataFrame(
    {
        "indicador": [
            "filas",
            "columnas",
            "fecha_minima",
            "fecha_maxima",
            "nulos_totales",
            "duplicados_fecha",
            "max_error_producto_inverso",
        ],
        "valor": [
            len(df),
            df.shape[1],
            df["fecha"].min().date().isoformat(),
            df["fecha"].max().date().isoformat(),
            int(df.isna().sum().sum()),
            int(df["fecha"].duplicated().sum()),
            float((df["uyu_por_brl"] * df["brl_por_uyu"] - 1).abs().max()),
        ],
    }
)

path_resumen = save_table(validation_summary, "resumen_dataset.csv")
path_nulos = save_table(df.isna().sum().reset_index().rename(columns={"index": "variable", 0: "nulos"}), "nulos.csv")

display(validation_summary)
display(df.head())
print(f"Resumen guardado en: {path_resumen}")
print(f"Nulos guardados en: {path_nulos}")



## 5. Estadistica descriptiva de UYU/BRL

Se calculan medidas de tendencia central, posicion, dispersion, forma y valores
atipicos por criterio IQR. Tambien se reportan las modas exactas, aunque en una
variable continua suelen tener baja utilidad descriptiva.


In [ ]:


def descriptive_statistics(values: pd.Series) -> tuple[pd.DataFrame, list[float], int]:
    series = clean_numeric(values)
    q1 = series.quantile(0.25)
    median = series.median()
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    outlier_mask = (series < lower_fence) | (series > upper_fence)
    mode_counts = series.value_counts()
    max_mode_count = int(mode_counts.max())
    modes = sorted(float(value) for value in mode_counts[mode_counts == max_mode_count].index)

    table = pd.DataFrame(
        [
            ("n", series.size),
            ("media", series.mean()),
            ("mediana", median),
            ("moda", modes[0]),
            ("q1", q1),
            ("q3", q3),
            ("iqr", iqr),
            ("varianza_muestral", series.var(ddof=1)),
            ("desvio_estandar_muestral", series.std(ddof=1)),
            ("minimo", series.min()),
            ("maximo", series.max()),
            ("rango", series.max() - series.min()),
            ("asimetria", series.skew()),
            ("curtosis", series.kurt()),
            ("limite_inferior_outliers", lower_fence),
            ("limite_superior_outliers", upper_fence),
            ("cantidad_outliers_iqr", int(outlier_mask.sum())),
        ],
        columns=["medida", "valor"],
    )
    return table, modes, max_mode_count


tabla_descriptiva, modas, frecuencia_moda = descriptive_statistics(df["uyu_por_brl"])
tabla_modas = pd.DataFrame({"moda_exacta": modas, "frecuencia": frecuencia_moda})
path_descriptiva = save_table(tabla_descriptiva, "descriptivo_uyu_por_brl.csv")

resumen_anual = (
    df.assign(anio=df["fecha"].dt.year)
    .groupby("anio", as_index=False)
    .agg(
        media_uyu_por_brl=("uyu_por_brl", "mean"),
        desvio_uyu_por_brl=("uyu_por_brl", "std"),
        minimo_uyu_por_brl=("uyu_por_brl", "min"),
        maximo_uyu_por_brl=("uyu_por_brl", "max"),
        observaciones=("uyu_por_brl", "count"),
    )
)
path_resumen_anual = save_table(resumen_anual, "resumen_anual.csv")

display(tabla_descriptiva)
display(tabla_modas)
display(resumen_anual.head())
print(f"Tabla descriptiva guardada en: {path_descriptiva}")
print(f"Resumen anual guardado en: {path_resumen_anual}")



## 6. Graficos principales

Se generan los graficos usados como apoyo del informe: serie temporal,
histograma, boxplot, Q-Q plot normal y relacion entre BRL/USD y UYU/USD.

No se genera matriz de correlacion.


In [ ]:

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(df["fecha"], df["uyu_por_brl"], color=PRIMARY_COLOR, linewidth=1)
ax.set_title("Evolucion diaria del tipo de cambio UYU/BRL")
ax.set_xlabel("Fecha")
ax.set_ylabel("UYU por BRL")
apply_common_style(ax)
path_serie = save_figure(fig, "serie_uyu_por_brl.png")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(x, bins=40, color=PRIMARY_COLOR, alpha=0.78, edgecolor="white")
ax.set_title("Distribucion empirica del tipo de cambio UYU/BRL")
ax.set_xlabel("UYU por BRL")
ax.set_ylabel("Frecuencia")
apply_common_style(ax)
path_hist = save_figure(fig, "hist_uyu_por_brl.png")

fig, ax = plt.subplots(figsize=(7, 3))
ax.boxplot(x, vert=False, patch_artist=True, boxprops={"facecolor": "#9ecae1"})
ax.set_title("Boxplot del tipo de cambio UYU/BRL")
ax.set_xlabel("UYU por BRL")
ax.set_yticks([])
apply_common_style(ax)
path_box = save_figure(fig, "boxplot_uyu_por_brl.png")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
stats.probplot(x, dist="norm", plot=ax)
ax.set_title("Q-Q plot de UYU/BRL frente a Normal")
apply_common_style(ax)
path_qq = save_figure(fig, "qq_uyu_por_brl.png")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["brl_por_usd"], df["uyu_por_usd"], s=12, alpha=0.45, color=ACCENT_COLOR, edgecolors="none")
ax.set_title("Relacion entre BRL/USD y UYU/USD")
ax.set_xlabel("BRL por USD")
ax.set_ylabel("UYU por USD")
apply_common_style(ax)
path_scatter = save_figure(fig, "scatter_brl_usd_vs_uyu_usd.png")



## 7. Analisis temporal y autocorrelacion

Se generan diagramas de rezago y autocorrelaciones seleccionadas de la cotizacion.
Esto permite ver dependencia temporal antes de interpretar inferencia clasica.


In [ ]:

lag_data = pd.DataFrame({"x_t": x})
lag_data["x_t_1"] = lag_data["x_t"].shift(1)
lag_data["x_t_2"] = lag_data["x_t"].shift(2)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(lag_data["x_t_1"], lag_data["x_t"], s=10, alpha=0.45, color=PRIMARY_COLOR, edgecolors="none")
ax.set_title("Diagrama de dispersion: X_t vs X_{t-1}")
ax.set_xlabel("UYU/BRL en t-1")
ax.set_ylabel("UYU/BRL en t")
apply_common_style(ax)
path_lag1 = save_figure(fig, "lag1_uyu_por_brl.png")

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(lag_data["x_t_2"], lag_data["x_t"], s=10, alpha=0.45, color=PRIMARY_COLOR, edgecolors="none")
ax.set_title("Diagrama de dispersion: X_t vs X_{t-2}")
ax.set_xlabel("UYU/BRL en t-2")
ax.set_ylabel("UYU/BRL en t")
apply_common_style(ax)
path_lag2 = save_figure(fig, "lag2_uyu_por_brl.png")


def autocorrelation_values(values: pd.Series, nlags: int = 30) -> pd.DataFrame:
    sample = clean_numeric(values).to_numpy(dtype=float)
    sample = sample - sample.mean()
    denominator = np.dot(sample, sample)
    autocorrelations = [1.0]
    for lag in range(1, nlags + 1):
        autocorrelations.append(float(np.dot(sample[:-lag], sample[lag:]) / denominator))
    return pd.DataFrame({"lag": range(nlags + 1), "autocorrelacion": autocorrelations})


acf_full = autocorrelation_values(df["uyu_por_brl"], 30)
acf_selected = acf_full[acf_full["lag"].isin([1, 2, 5, 10, 20, 30])].reset_index(drop=True)
path_acf_table = save_table(acf_selected, "autocorrelacion_uyu_por_brl.csv")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(acf_full["lag"], acf_full["autocorrelacion"], color=BAR_COLOR)
ax.set_title("Autocorrelacion de la cotizacion UYU/BRL")
ax.set_xlabel("Rezago")
ax.set_ylabel("Autocorrelacion")
ax.set_ylim(0, 1.05)
apply_common_style(ax)
path_acf = save_figure(fig, "acf_uyu_por_brl.png")

display(acf_selected)
print(f"Autocorrelaciones guardadas en: {path_acf_table}")



## 8. Correlacion de Pearson descriptiva

Se calcula Pearson como medida descriptiva de asociacion lineal. No se interpreta
como causalidad. La variable UYU/BRL se construye algebraicamente a partir de las
cotizaciones base, por lo que hay que ser cuidadoso con la lectura economica.


In [ ]:


def pearson_descriptive(dataframe: pd.DataFrame, pairs: list[tuple[str, str]]) -> pd.DataFrame:
    rows = []
    for x_column, y_column in pairs:
        pair = dataframe[[x_column, y_column]].dropna()
        rows.append(
            {
                "variable_x": x_column,
                "variable_y": y_column,
                "pearson_r": pair[x_column].corr(pair[y_column]),
                "n": len(pair),
            }
        )
    return pd.DataFrame(rows)


correlaciones = pearson_descriptive(
    df,
    [
        ("brl_por_usd", "uyu_por_usd"),
        ("brl_por_usd", "uyu_por_brl"),
        ("uyu_por_usd", "uyu_por_brl"),
        ("brl_por_uyu", "uyu_por_brl"),
    ],
)
path_correlaciones = save_table(correlaciones, "correlaciones.csv")

display(correlaciones)
print(f"Correlaciones guardadas en: {path_correlaciones}")



## 9. Bondad de ajuste chi-cuadrado

Se comparan Normal, Log-normal y Gamma mediante prueba chi-cuadrado de bondad de
ajuste con 12 clases equiprobables bajo cada modelo. La mejor distribucion se
elige en sentido relativo por menor estadistico chi-cuadrado.


In [ ]:


def chi_square_distribution_fit(values: pd.Series, bins: int = 12) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    sample = clean_numeric(values)
    n = len(sample)
    candidates = {
        "Normal": (stats.norm, stats.norm.fit(sample), 2),
        "Log-normal": (stats.lognorm, stats.lognorm.fit(sample, floc=0), 3),
        "Gamma": (stats.gamma, stats.gamma.fit(sample, floc=0), 3),
    }
    rows = []
    class_tables = {}

    for name, (distribution, params, estimated_params) in candidates.items():
        edges = distribution.ppf(np.linspace(0, 1, bins + 1), *params)
        edges[0] = -np.inf
        edges[-1] = np.inf
        observed, _ = np.histogram(sample, bins=edges)
        expected = np.full(bins, n / bins)
        chi2 = float(((observed - expected) ** 2 / expected).sum())
        dof = bins - 1 - estimated_params
        p_value = float(stats.chi2.sf(chi2, dof))
        rows.append(
            {
                "distribucion": name,
                "clases": bins,
                "parametros_estimados": estimated_params,
                "frecuencia_esperada_minima": expected.min(),
                "grados_libertad": dof,
                "chi2_estadistico": chi2,
                "p_value": p_value,
                "decision_alpha_0_05": "Se rechaza H0" if p_value < 0.05 else "No se rechaza H0",
            }
        )
        class_tables[name] = pd.DataFrame(
            {
                "clase": range(1, bins + 1),
                "limite_inferior": edges[:-1],
                "limite_superior": edges[1:],
                "frecuencia_observada": observed,
                "frecuencia_esperada": expected,
                "aporte_chi2": (observed - expected) ** 2 / expected,
            }
        )

    return pd.DataFrame(rows).sort_values("chi2_estadistico"), class_tables


bondad_chi2, tablas_clases = chi_square_distribution_fit(df["uyu_por_brl"], bins=12)
path_bondad = save_table(bondad_chi2, "bondad_ajuste_chi2_uyu_por_brl.csv")
path_clases_lognormal = save_table(tablas_clases["Log-normal"], "bondad_ajuste_chi2_clases_lognormal.csv")

display(bondad_chi2)
display(tablas_clases["Log-normal"])
print(f"Bondad de ajuste guardada en: {path_bondad}")
print(f"Clases Log-normal guardadas en: {path_clases_lognormal}")



## 10. Intervalos de confianza

Se calculan intervalos clasicos para la media mediante t de Student y para la
varianza mediante chi-cuadrado. Tambien se calcula un bootstrap percentil para la
media como comparacion empirica.


In [ ]:

CONFIDENCE_LEVELS = (0.90, 0.95, 0.99)


def confidence_interval_mean(values: pd.Series, confidence: float) -> dict[str, float | int | str]:
    sample = clean_numeric(values).to_numpy(dtype=float)
    n = sample.size
    mean = float(sample.mean())
    se = float(sample.std(ddof=1) / np.sqrt(n))
    alpha = 1 - confidence
    critical = float(stats.t.ppf(1 - alpha / 2, df=n - 1))
    margin = critical * se
    return {
        "nivel_confianza": confidence,
        "media": mean,
        "limite_inferior": mean - margin,
        "limite_superior": mean + margin,
        "amplitud": 2 * margin,
        "metodo": "t de Student",
    }


def confidence_interval_variance(values: pd.Series, confidence: float) -> dict[str, float | int | str]:
    sample = clean_numeric(values).to_numpy(dtype=float)
    n = sample.size
    s2 = float(sample.var(ddof=1))
    alpha = 1 - confidence
    lower = (n - 1) * s2 / stats.chi2.ppf(1 - alpha / 2, df=n - 1)
    upper = (n - 1) * s2 / stats.chi2.ppf(alpha / 2, df=n - 1)
    return {
        "nivel_confianza": confidence,
        "varianza_muestral": s2,
        "limite_inferior": float(lower),
        "limite_superior": float(upper),
        "amplitud": float(upper - lower),
        "metodo": "chi-cuadrado",
    }


def bootstrap_mean_interval(values: pd.Series, confidence: float = 0.95, n_boot: int = 5000, seed: int = 42) -> pd.DataFrame:
    sample = clean_numeric(values).to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    boot_means = rng.choice(sample, size=(n_boot, sample.size), replace=True).mean(axis=1)
    alpha = 1 - confidence
    lower = np.quantile(boot_means, alpha / 2)
    upper = np.quantile(boot_means, 1 - alpha / 2)
    return pd.DataFrame(
        [
            {
                "nivel_confianza": confidence,
                "media_bootstrap": boot_means.mean(),
                "limite_inferior": lower,
                "limite_superior": upper,
                "amplitud": upper - lower,
                "metodo": "bootstrap percentil",
                "repeticiones": n_boot,
                "semilla": seed,
            }
        ]
    )


ic_media = pd.DataFrame([confidence_interval_mean(df["uyu_por_brl"], level) for level in CONFIDENCE_LEVELS])
ic_varianza = pd.DataFrame([confidence_interval_variance(df["uyu_por_brl"], level) for level in CONFIDENCE_LEVELS])
bootstrap_media = bootstrap_mean_interval(df["uyu_por_brl"])

path_ic_media = save_table(ic_media, "ic_media_uyu_por_brl.csv")
path_ic_varianza = save_table(ic_varianza, "ic_varianza_uyu_por_brl.csv")
path_bootstrap = save_table(bootstrap_media, "bootstrap_media_uyu_por_brl.csv")

display(ic_media)
display(ic_varianza)
display(bootstrap_media)

print(f"IC media guardado en: {path_ic_media}")
print(f"IC varianza guardado en: {path_ic_varianza}")
print(f"Bootstrap media guardado en: {path_bootstrap}")



## 11. Resumen de archivos creados o actualizados

Esta lista permite verificar rapidamente que las tablas y figuras principales
fueron generadas. Se excluye explicitamente la matriz de correlacion.


In [ ]:

created_or_updated = [
    TABLES_DIR / "resumen_dataset.csv",
    TABLES_DIR / "nulos.csv",
    TABLES_DIR / "descriptivo_uyu_por_brl.csv",
    TABLES_DIR / "resumen_anual.csv",
    TABLES_DIR / "autocorrelacion_uyu_por_brl.csv",
    TABLES_DIR / "correlaciones.csv",
    TABLES_DIR / "bondad_ajuste_chi2_uyu_por_brl.csv",
    TABLES_DIR / "bondad_ajuste_chi2_clases_lognormal.csv",
    TABLES_DIR / "ic_media_uyu_por_brl.csv",
    TABLES_DIR / "ic_varianza_uyu_por_brl.csv",
    TABLES_DIR / "bootstrap_media_uyu_por_brl.csv",
    FIGURES_DIR / "serie_uyu_por_brl.png",
    FIGURES_DIR / "hist_uyu_por_brl.png",
    FIGURES_DIR / "boxplot_uyu_por_brl.png",
    FIGURES_DIR / "qq_uyu_por_brl.png",
    FIGURES_DIR / "scatter_brl_usd_vs_uyu_usd.png",
    FIGURES_DIR / "lag1_uyu_por_brl.png",
    FIGURES_DIR / "lag2_uyu_por_brl.png",
    FIGURES_DIR / "acf_uyu_por_brl.png",
]

for path in created_or_updated:
    status = "OK" if path.exists() else "FALTA"
    print(f"{status}: {path}")



## 12. Cierre metodologico

La Log-normal fue la mejor distribucion relativa entre las candidatas evaluadas
por chi-cuadrado, seguida por la Gamma. Como los p-values fueron menores que
0,05, ninguna distribucion se presenta como ajuste exacto. Ademas, la serie
muestra autocorrelacion alta, por lo que los intervalos clasicos deben
interpretarse con cautela.

Correlacion no implica causalidad.
